# Privacy Risk of Synthetic Data

## What This Is
Synthetic data is often assumed to be privacy-safe because it contains no real records. This is wrong. Privacy risks in synthetic data include:

1. **Membership inference on synthetic data**: Can an attacker determine whether a specific individual was in the training set used to generate the synthetic data?
2. **Linkage attacks**: Combine synthetic data with auxiliary information to re-identify individuals
3. **Attribute inference**: Infer sensitive attributes of real individuals from statistical properties of synthetic data
4. **Memorization in the generator**: Deep generative models can memorize and reproduce training samples verbatim

## Membership Inference on Synthetic Data
The attack works by training a shadow generator on known real/non-member data, then using the shadow model's outputs to train a classifier that distinguishes member vs non-member query behavior.

For simpler generators (Gaussian copulas), the attack reduces to checking how closely the synthetic samples match the query: if the synthetic data distribution closely reflects a specific individual, they were likely in the training set.

In [ ]:
import numpy as np
from typing import Dict, List, Tuple

np.random.seed(42)

# Privacy risk analysis for synthetic data generators

def fit_gaussian_generator(X_train: np.ndarray):
    """Fit a simple Gaussian generator (mu, cov)."""
    mu = X_train.mean(axis=0)
    cov = np.cov(X_train.T) + np.eye(X_train.shape[1]) * 0.01
    return mu, cov

def generate_gaussian(mu, cov, n: int) -> np.ndarray:
    return np.random.multivariate_normal(mu, cov, n)

def gaussian_log_likelihood(x: np.ndarray, mu: np.ndarray, cov: np.ndarray) -> float:
    """Log likelihood of x under Gaussian(mu, cov)."""
    d = len(mu)
    diff = x - mu
    sign, logdet = np.linalg.slogdet(cov)
    if sign <= 0: return -np.inf
    inv_cov = np.linalg.inv(cov)
    return -0.5 * (d * np.log(2 * np.pi) + logdet + diff @ inv_cov @ diff)

# Dataset: 200 members, 200 non-members
N_TOTAL = 200
D = 5
X_members = np.random.randn(N_TOTAL, D)  # training data
X_members[:, 0] += 1  # slight distribution shift from non-members
X_nonmembers = np.random.randn(N_TOTAL, D)

# Train generator on members only
mu, cov = fit_gaussian_generator(X_members)

# Likelihood-based membership inference
# High likelihood under the generator = more likely member
member_lls = np.array([gaussian_log_likelihood(x, mu, cov) for x in X_members])
nonmember_lls = np.array([gaussian_log_likelihood(x, mu, cov) for x in X_nonmembers])

print(f'Average log-likelihood (members):     {member_lls.mean():.4f}')
print(f'Average log-likelihood (non-members): {nonmember_lls.mean():.4f}')
print(f'Gap: {member_lls.mean() - nonmember_lls.mean():.4f}')


In [ ]:
# MIA evaluation for synthetic data
from sklearn.metrics import roc_auc_score

# Use likelihood ratio as attack score
all_scores = np.concatenate([member_lls, nonmember_lls])
all_labels = np.array([1]*N_TOTAL + [0]*N_TOTAL)

mia_auc = roc_auc_score(all_labels, all_scores)

# Threshold-based attack accuracy
threshold = np.median(all_scores)
preds = (all_scores > threshold).astype(int)
attack_acc = (preds == all_labels).mean()

print('Membership Inference Attack on Synthetic Data Generator')
print('=' * 55)
print(f'MIA AUC:      {mia_auc:.3f} (0.5 = random, 1.0 = perfect)')
print(f'Attack Acc:   {attack_acc:.3f} (0.5 = random)')

print('\nInterpretation:')
if mia_auc > 0.7:
    print('  HIGH privacy risk: synthetic data generator leaks membership information')
elif mia_auc > 0.55:
    print('  MODERATE privacy risk: some membership signal present')
else:
    print('  LOW privacy risk: membership near random')

print('\nNote: Adding DP-SGD during generator training would reduce MIA AUC toward 0.5')


In [ ]:
# DCR privacy audit: are any synthetic samples too close to real training samples?
synth_samples = generate_gaussian(mu, cov, 500)

def compute_dcr_audit(synth: np.ndarray, real: np.ndarray, 
                       threshold_pct: float = 5) -> Dict:
    """Compute DCR and flag synthetic samples that are dangerously close to real ones."""
    min_dists = []
    for s in synth:
        dists = np.linalg.norm(real - s, axis=1)
        min_dists.append(dists.min())
    min_dists = np.array(min_dists)
    
    # Baseline: distance between random real samples (expected minimum)
    baseline_dists = []
    for i in range(min(200, len(real))):
        dists = np.linalg.norm(real - real[i], axis=1)
        dists[i] = np.inf
        baseline_dists.append(dists.min())
    baseline_5pct = np.percentile(baseline_dists, threshold_pct)
    
    high_risk = (min_dists < baseline_5pct * 0.5).sum()
    
    return {
        'dcr_mean': min_dists.mean(),
        'dcr_5pct': np.percentile(min_dists, 5),
        'baseline_5pct': baseline_5pct,
        'high_risk_samples': high_risk,
        'high_risk_pct': high_risk / len(synth) * 100,
    }

audit = compute_dcr_audit(synth_samples, X_members)

print('DCR Privacy Audit')
print('=' * 45)
for k, v in audit.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

print('\nHigh-risk samples: synthetic records < 50% of baseline minimum distance')
print('These may represent near-copies of real training records.')
